In [76]:
# !pip install ta-lib
# !pip install gdown
# !pip install requests
# !pip install numpy
# !pip install pandas


In [77]:
# !pip install -r requirements_dev.txt       
# !pip install pyotp
# !pip install logzero
# !pip install websocket-client    

In [78]:
# !pip uninstall pycrypto
# !pip install pycryptodome    

In [79]:
import pandas as pd
# import numpy as np
import requests
from datetime import datetime
import socket
import uuid
# import http.client
import time
import requests # type: ignore
# import mimetypes
import json
import talib
# import gdown
import http
import ssl
import os

In [80]:
# !pip install python-dotenv

In [81]:
from SmartApi import SmartConnect #or from SmartApi.smartConnect import SmartConnect
import pyotp
from logzero import logger
# from dotenv import load_dotenv

In [82]:
# load_dotenv()

In [83]:
# Static values
user_type = "USER"
source_id = "WEB"
# api_key = os.getenv("ANG_ONE_KEY")   
api_key = os.environ["ANG_ONE_KEY"]  
# client_code = os.getenv("CLIENTCODE")
client_code = os.environ["CLIENTCODE"]
# password = os.getenv("PASSWORD")
password = os.environ["PASSWORD"]
window = 965

# todays_date = datetime.today().strftime("%Y-%m-%d")
todays_date = (datetime.today() - pd.DateOffset(days=0)).strftime("%Y-%m-%d")
window_date = (datetime.today() - pd.DateOffset(days=window)).strftime("%Y-%m-%d")

In [84]:
local_ip = socket.gethostbyname(socket.gethostname())
smartApi = SmartConnect(api_key)

try:
    # token = os.getenv("TOTP_TOKEN")
    token = os.environ["TOTP_TOKEN"]
    totp = pyotp.TOTP(token).now()
except Exception as e:
    logger.error("Invalid Token: The provided token is not valid.")
    raise e


# Get Public IP
public_ip = requests.get('https://api.ipify.org').text

# Get MAC Address
mac_address = ':'.join(['{:02x}'.format((uuid.getnode() >> ele) & 0xff)
                        for ele in range(0,8*6,8)][::-1])

# Change clientcode, password, totp
payload = '''{\n\"clientcode\":\"'''+str(client_code)+'''\"
         ,\n\"password\":\"'''+str(password)+'''\"\n
		,\n\"totp\":\"'''+str(totp)+'''\"\n
    ,\n\"state\":\"Active\"\n}'''

headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    "X-UserType": user_type,
    "X-SourceID": source_id,
    "X-ClientLocalIP": local_ip,
    "X-ClientPublicIP": public_ip,
    "X-MACAddress": mac_address,
    'X-PrivateKey': api_key #'QNeuDKb5'
}


context = ssl._create_unverified_context()

conn = http.client.HTTPSConnection(
    "apiconnect.angelone.in", context=context
    )

conn.request("POST", "/rest/auth/angelbroking/user/v1/loginByPassword", payload, headers)
time.sleep(0.5)
res = conn.getresponse()
data = res.read()
data = data.decode("utf-8")

[I 251106 12:53:11 smartConnect:121] in pool


In [85]:
temp = json.loads(data)
jwtToken = temp["data"]["jwtToken"]
print(jwtToken)

# user_type = "USER"
# source_id = "WEB"
local_ip = socket.gethostbyname(socket.gethostname())
public_ip = requests.get('https://api.ipify.org').text
mac_address = ':'.join(['{:02x}'.format((uuid.getnode() >> ele) & 0xff)
                        for ele in range(0,8*6,8)][::-1])
authToken = f'Bearer {jwtToken}'


headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    "X-UserType": user_type,
    "X-SourceID": source_id,
    "X-ClientLocalIP": local_ip,
    "X-ClientPublicIP": public_ip,
    "X-MACAddress": mac_address,
    'X-PrivateKey': api_key,
    'Authorization': authToken ,
}


eyJhbGciOiJIUzUxMiJ9.eyJ1c2VybmFtZSI6IkJHQkcxMTQ0Iiwicm9sZXMiOjAsInVzZXJ0eXBlIjoiVVNFUiIsInRva2VuIjoiZXlKaGJHY2lPaUpTVXpJMU5pSXNJblI1Y0NJNklrcFhWQ0o5LmV5SjFjMlZ5WDNSNWNHVWlPaUpqYkdsbGJuUWlMQ0owYjJ0bGJsOTBlWEJsSWpvaWRISmhaR1ZmWVdOalpYTnpYM1J2YTJWdUlpd2laMjFmYVdRaU9qRXhMQ0p6YjNWeVkyVWlPaUl6SWl3aVpHVjJhV05sWDJsa0lqb2lZMlZrWkRreU9XWXRaV1ZsWkMwek1ERmlMV0k1TldVdE9EZ3lZVEk1TkdVM01EQTFJaXdpYTJsa0lqb2lkSEpoWkdWZmEyVjVYM1l5SWl3aWIyMXVaVzFoYm1GblpYSnBaQ0k2TVRFc0luQnliMlIxWTNSeklqcDdJbVJsYldGMElqcDdJbk4wWVhSMWN5STZJbUZqZEdsMlpTSjlMQ0p0WmlJNmV5SnpkR0YwZFhNaU9pSmhZM1JwZG1VaWZYMHNJbWx6Y3lJNkluUnlZV1JsWDJ4dloybHVYM05sY25acFkyVWlMQ0p6ZFdJaU9pSkNSMEpITVRFME5DSXNJbVY0Y0NJNk1UYzJNalV4T1RrNU5Dd2libUptSWpveE56WXlORE16TkRFMExDSnBZWFFpT2pFM05qSTBNek0wTVRRc0ltcDBhU0k2SW1FME5qZzFPVEZsTFRaaU1tSXRORFUzT0MwNVlqYzVMVGd4WWprNVl6SXhZelE1TXlJc0lsUnZhMlZ1SWpvaUluMC5XTnI3SkhYbFFDZVUzX1UyR3V0TXRwVHpjb1NZZ01CSlhuTV9oWEtvRkdCVlcyc2dXSFBCYXZPMTBPcnMwbWZIRkllRjRBLUJ2LTRPS21YcXlmNnJiSHZVX3FxekQ2d0IxT0x6bk9XM2VWSFhSWkZoWnhGV2R

In [86]:
# shareable_link = 'https://drive.google.com/file/d/1PdYMxjWQ4tBJp4Mmp1LjLOR2H2vkZ6on/view?usp=sharing'

# Extract the file ID
# file_id = shareable_link.split('/d/')[1].split('/view')[0]

# Construct the download URL
# download_url = f'https://drive.google.com/uc?id={file_id}'


# # Download the file using gdown
# output_file = 'Nifty500-token.csv'

# stock_symbols_df = pd.read_csv(output_file)
# stock_symbols_df["token"] = stock_symbols_df["token"].fillna(891)
# stock_symbols_df["token"] = stock_symbols_df["token"].astype(int)
# stocks_to_consider = 'PO1_Stocks.csv'

# stock_lists = pd.read_csv(stocks_to_consider)
# main_df = pd.merge(stock_lists[['rsi','symbol','win_ratio','priority']], stock_symbols_df[['Symbol','token']], left_on ='symbol' , right_on='Symbol', how='inner')

# main_df.to_csv('Main_df.csv', index=False)

In [87]:

# # Download the file using gdown
# output_file = 'Nifty500-token.csv'
# stock_symbols_df = pd.read_csv(output_file)
# stock_symbols_df["token"] = stock_symbols_df["token"].fillna(891)
# stock_symbols_df["token"] = stock_symbols_df["token"].astype(int)
# stocks_to_consider = 'Stocks.csv'

# stock_lists = pd.read_csv(stocks_to_consider)
# full_main_df = pd.merge(stock_lists[['rsi','symbol','win_ratio','priority']], stock_symbols_df[['Symbol','token']], left_on ='symbol' , right_on='Symbol', how='inner')


# # full_main_df.to_csv('Full_Main_df.csv', index=False)

# main_df = full_main_df.copy()

# full_main_df = pd.read_csv('Full_Main_df.csv')

In [88]:
main_df = pd.read_csv('Main_df.csv')

In [89]:
# print(main_df)

In [90]:
# Step 2: Function to fetch daily candle data from API
def fetch_candle_data(symbol,interval='ONE_DAY'):
  
    payload = '''{\r\n     \"exchange\": \"NSE\",\r\n
          \"symboltoken\": \"'''+str(symbol)+'''\",\r\n     \"interval\": \"'''+str(interval)+'''\",\r\n
          \"fromdate\": \"'''+str(window_date)+''' 16:30\",\r\n     \"todate\": \"'''+str(todays_date)+''' 16:30\"\r\n}
    '''
    # payload = '''{\r\n     \"exchange\": \"NSE\",\r\n
    #       \"symboltoken\": \"'''+str(symbol)+'''\",\r\n     \"interval\": \"ONE_DAY\",\r\n
    #       \"fromdate\": \"2025-09-01 16:30\",\r\n     \"todate\": \"2025-12-31 16:30\"\r\n}
    # '''

    conn = http.client.HTTPSConnection("apiconnect.angelone.in", context=context)
    conn.request("POST", "/rest/secure/angelbroking/historical/v1/getCandleData", payload, headers)
    res = conn.getresponse()
    data = res.read()
    data = data.decode("utf-8")
    json_data = json.loads(data)
    json_data = json_data['data']
    # print(json_data)
    return json_data

# Step 4: Function to calculate RSI trends (increase or decrease)
def rsi_trend1(rsi_values, setup_rsi):
    if rsi_values[-1] >= 60 and  rsi_values[-2] < 60:
      return "60 CROSSOVER"

    if rsi_values[-1] >= setup_rsi and  rsi_values[-2] < setup_rsi:
      return setup_rsi + 1

    if rsi_values[-1] < setup_rsi and rsi_values[-2] >= setup_rsi:
      return setup_rsi - 1

    if rsi_values[-1] > rsi_values[-2]:
        return 'UP'
    else:
        return 'DOWN'
    

# Fuction to identify RSI Breakout
def rsi_trend(rsi_values, setup_rsi):
    if rsi_values[-1] >= setup_rsi and  rsi_values[-2] < setup_rsi:
      return True
    

# Fuction to identify stocks before RSI Breakout 
def rsi_trend_P1(rsi_values, setup_rsi):    
    if rsi_values[-1] >= ( setup_rsi - 5 ) and  rsi_values[-1] < ( setup_rsi + 5):
      return True
     
def fetch_candle_week_month_data(symbol,range='W',daily_json_data=None):
    
    if daily_json_data is None:
      daily_json_data = fetch_candle_data(symbol)
      
    df = pd.DataFrame(daily_json_data, columns=["Date", "Open", "High", "Low", "Close", "Volume"])
    # Convert 'Date' column to datetime
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

    # Make Date the DatetimeIndex required by resample
    df.set_index('Date', inplace=True)
    
    # # Sort data by date in ascending order
    # df = df.sort_values('Date').reset_index(drop=True)

    range_data = df.resample(range).agg({
    # 'Date' : 'last',
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
      }).dropna()
    
    

    range_data['RSI_14'] = talib.RSI(range_data['Close'], timeperiod=14)
    return range_data
    # range_data.set_index('Date', inplace=True)
    

    

In [91]:
# print("Fetching daily data ")
# print( fetch_candle_data(438) )
# print("Fetching weekly data ")
# print( fetch_candle_week_month_data(395,'W') )

In [92]:
# Prepare output DataFrame
output_data = []
error_data = []
priority_data = []
priority_watch_data = []
priority0_data = []
rsi_40_cross_data = []

print(main_df.columns.tolist())
# main_df.sort_values(by=['symbol'], ascending=[False, False], inplace=True)

# Step 5: Process each stock
for _, row in main_df.iterrows():

    time.sleep(0.4)

    # company = row['NAME OF COMPANY']
    priority = row['priority']
    name = row['Symbol']
    token = row['token']
    rsi = row['rsi']
    win_ratio = row['win_ratio']
    try:

        if rsi == None:
            error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': 'RSI Not Maintained in file'
             }) 
            continue
   
   
        # Fetch daily data
        daily_json_data = fetch_candle_data(token)
        if daily_json_data == None:
            error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': 'Error while fetching data'
             })    
            continue
        # print(daily_json_data)

        monthly_data = fetch_candle_week_month_data(token,'W',daily_json_data)   
        
        # print(monthly_data)
        logger.info(f"Processing {name} with token {token} and RSI {rsi}")
        last_2_rsi_monthly = monthly_data['RSI_14'].dropna().tail(2).values
        
        if len(last_2_rsi_monthly) < 2:
             error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': 'Not enough Monthly RSI data'
             })
             continue
         
        if priority == 2 and rsi_trend1(last_2_rsi_monthly,40) == "39":
            rsi_40_cross_data.append({
                'Name': name,
                'Token': token,
                'Monthly_RSI': last_2_rsi_monthly[-1],
                'last_Month_RSI': last_2_rsi_monthly[-2],
                'Priority': priority,
            })


        df = pd.DataFrame(daily_json_data, columns=["Date", "Open", "High", "Low", "Close", "Volume"])
        
        # break

        # Convert 'Date' column to datetime
        df['Date'] = pd.to_datetime(df['Date'])

        # Sort data by date in ascending order
        df = df.sort_values('Date').reset_index(drop=True)


        df['RSI_14'] = talib.RSI(df['Close'], timeperiod=14)
        df.set_index('Date', inplace=True)
                
        last_2_rsi_daily = df['RSI_14'].dropna().tail(2).values

        
        # if token == 438:
        #     print(last_2_rsi_daily)

        if len(last_2_rsi_daily) < 2:
             error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': 'Not enough RSI data'
             })
             continue
        daily_rsi = last_2_rsi_daily[-1]

        last_dats = daily_json_data[-1][0].split('T')[0]
        
        if last_dats != todays_date:
             error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': f'Date from API {last_dats}, processing date {todays_date}'
             })
             continue


        if priority == 2:
            if rsi_trend(last_2_rsi_daily, rsi):
                priority_data.append({
                    'Name': name,
                    'Token': token,
                    'Setup_RSI': rsi,
                    'Daily_RSI': daily_rsi,
                    'yesterday_RSI': last_2_rsi_daily[-2],
                    'win_ratio': win_ratio,

                })

            if rsi_trend_P1(last_2_rsi_daily, rsi):
                priority_watch_data.append({
                    'Name': name,
                    'Token': token,
                    'Setup_RSI': rsi,
                    'Daily_RSI': daily_rsi,
                    'yesterday_RSI': last_2_rsi_daily[-2],
                    'win_ratio': win_ratio,

                })
                
        elif priority == 1 and rsi_trend(last_2_rsi_daily, rsi):
            output_data.append({
                'Name': name,
                'Token': token,
                'Setup_RSI': rsi,
                'Daily_RSI': daily_rsi,
                'yesterday_RSI': last_2_rsi_daily[-2],
                'win_ratio': win_ratio,

            })
        elif priority == 0 and rsi_trend(last_2_rsi_daily, rsi):
            priority0_data.append({
                'Name': name,
                'Token': token,
                'Setup_RSI': rsi,
                'Daily_RSI': daily_rsi,
                'yesterday_RSI': last_2_rsi_daily[-2],
                'win_ratio': win_ratio,

            })
           
                      

        

    except Exception as e:
        error_data.append({
            'Name': name,
            'Token': token,
            'Setup_RSI': rsi,
            'reasone': str(e)
        })
        print(f"Error processing {name}: {e}")

['rsi', 'symbol', 'win_ratio', 'priority', 'Symbol', 'token']


[I 251106 12:53:15 4041550269:50] Processing 360ONE with token 13062 and RSI 42
[I 251106 12:53:16 4041550269:50] Processing ACI with token 12030 and RSI 50
[I 251106 12:53:16 4041550269:50] Processing ALKEM with token 11703 and RSI 46
[I 251106 12:53:17 4041550269:50] Processing ANANDRATHI with token 7145 and RSI 56
[I 251106 12:53:17 4041550269:50] Processing ANURAS with token 2829 and RSI 74
[I 251106 12:53:18 4041550269:50] Processing APOLLOHOSP with token 12209 and RSI 53
[I 251106 12:53:18 4041550269:50] Processing AXISBANK with token 5900 and RSI 49
[I 251106 12:53:18 4041550269:50] Processing BAJAJ-AUTO with token 16669 and RSI 48
[I 251106 12:53:19 4041550269:50] Processing BAJAJHLDNG with token 305 and RSI 42
[I 251106 12:53:19 4041550269:50] Processing BEML with token 395 and RSI 47
[I 251106 12:53:20 4041550269:50] Processing BHARTIARTL with token 10604 and RSI 48
[I 251106 12:53:20 4041550269:50] Processing BHEL with token 438 and RSI 60
[I 251106 12:53:21 4041550269:50] P

In [93]:
print(main_df)

    rsi      symbol win_ratio  priority      Symbol  token
0    42      360ONE    62.50%         1      360ONE  13062
1    50         ACI    69.44%         0         ACI  12030
2    46       ALKEM    63.01%         2       ALKEM  11703
3    56  ANANDRATHI    68.63%         0  ANANDRATHI   7145
4    74      ANURAS    87.50%         0      ANURAS   2829
..  ...         ...       ...       ...         ...    ...
68   51  TORNTPHARM    44.30%         0  TORNTPHARM   3518
69   55    TVSMOTOR    65.75%         1    TVSMOTOR   8479
70   51  ULTRACEMCO    62.07%         1  ULTRACEMCO  11532
71   52    UNITDSPR    70.31%         0    UNITDSPR  10447
72   57   ZYDUSLIFE    67.74%         0   ZYDUSLIFE   7929

[73 rows x 6 columns]


In [94]:

# priority_data.sort_values(by=['Name'], ascending=False, inplace=True)
# priority_watch_data.sort_values(by=['Name'], ascending=False, inplace=True)
# output_data.sort_values(by=['Name'], ascending=False, inplace=True)
# error_data.sort_values(by=['Name'], ascending=False, inplace=True)
# priority0_data.sort_values(by=['Name'], ascending=False, inplace=True)
# rsi_40_cross_data.sort_values(by=['Name'], ascending=False, inplace=True)

print("Priority Data:")
print(priority_data)    
print("Priority Watch Data:")
print(priority_watch_data)
print("Output Data:")
print(output_data)  
print("Error Data:")
print(error_data)


Priority Data:
[{'Name': 'BAJAJHLDNG', 'Token': 305, 'Setup_RSI': 42, 'Daily_RSI': np.float64(59.299858912059065), 'yesterday_RSI': np.float64(38.09205070090946), 'win_ratio': '64.00%'}, {'Name': 'INDIGO', 'Token': 11195, 'Setup_RSI': 43, 'Daily_RSI': np.float64(46.16948252782629), 'yesterday_RSI': np.float64(41.94290790231183), 'win_ratio': '66.67%'}]
Priority Watch Data:
[{'Name': 'HINDALCO', 'Token': 1363, 'Setup_RSI': 51, 'Daily_RSI': np.float64(46.38473471068718), 'yesterday_RSI': np.float64(63.82044281984138), 'win_ratio': '74.63%'}, {'Name': 'ICICIBANK', 'Token': 12458, 'Setup_RSI': 36, 'Daily_RSI': np.float64(32.03764606582257), 'yesterday_RSI': np.float64(35.88312174455883), 'win_ratio': '65.06%'}, {'Name': 'INDIGO', 'Token': 11195, 'Setup_RSI': 43, 'Daily_RSI': np.float64(46.16948252782629), 'yesterday_RSI': np.float64(41.94290790231183), 'win_ratio': '66.67%'}, {'Name': 'LT', 'Token': 11483, 'Setup_RSI': 53, 'Daily_RSI': np.float64(53.772650541451114), 'yesterday_RSI': np.fl

In [95]:
import requests
# import urllib.parse

BOT_TOKEN = "8446280700:AAEVJcAw73988-gAx8kJF1TKFMLwHVCM-gs"

TEST_ID = "529251493"
CHAT_ID = TEST_ID
# CHAT_ID = "-1003139839259"
def format_whatsapp_report(data ,name):
    
    lines = [f"📊 <b>{name}</b>"]
    
    if len(data) == 0:
        lines.append( "\n🔹 <b>No Stocks</b>" )
    else:
        for index,item in enumerate(data):
            lines.append(
                f"\n🔹 <b>{index+1} {item['Name']}</b>"
                f"\n   <b>Todays_RSI: </b>{item['Daily_RSI']:.2f}"
                f"\n   <b>Yesterdays_RSI: </b>{item['yesterday_RSI']:.2f}"
                f"\n   <b>Standard_RSI: </b>{item['Setup_RSI']:.2f}"
                # f"\n   High Priority: {'✅' if item['High Priority'] else '❌'}"
            )      
           
    # return urllib.parse.quote_plus( "\n".join(lines) )
    return "\n".join(lines) 

def format_whatsapp_error(data ,name):
    
    lines = [f"📊 <b>{name}</b>"]
    
    if len(data) == 0:
        lines.append( "\n🔹 <b>No Stocks</b>" )
    else:
        for index,item in enumerate(data):
            lines.append(
                f"\n🔹 <b>{index+1} {item['Name']}</b>"
                f"\n   <b>Token: </b>{item['Token']}"
                f"\n   <b>Standard_RSI: </b>{item['Setup_RSI']:.2f}"
                f"\n   <b>Reasone: </b>{item['reasone']}"
                # f"\n   High Priority: {'✅' if item['High Priority'] else '❌'}"
            )      
           
    # return urllib.parse.quote_plus( "\n".join(lines) )
    return "\n".join(lines)

def format_whatsapp_40_report(data ,name):
    
    lines = [f"📊 <b>{name}</b>"]
    
    if len(data) == 0:
        lines.append( "\n🔹 <b>No Stocks</b>" )
    else:
        for index,item in enumerate(data):
            lines.append(
                f"\n🔹 <b>{index+1} {item['Name']}</b>"
                f"\n   <b>Monthly_RSI: </b>{item['Monthly_RSI']:.2f}"
                f"\n   <b>Last_Month_RSI: </b>{item['last_Month_RSI']:.2f}"
                f"\n   <b>Priority: </b>{item['Priority']}"
            )      
           
    # return urllib.parse.quote_plus( "\n".join(lines) )
    return "\n".join(lines) 

In [96]:
# Telegram Message trigger logic
msg = f"📊 <b>Daily Report: {todays_date}</b>"
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
             params={"chat_id": CHAT_ID, "text": msg, "parse_mode": "HTML"})


msg_p1 = format_whatsapp_report(priority_data,'Priority Stocks')
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
             params={"chat_id": CHAT_ID, "text": msg_p1, "parse_mode": "HTML"})


msg_p2 = format_whatsapp_report(priority_watch_data,'Priority Stocks to be tracked')
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
             params={"chat_id": CHAT_ID, "text": msg_p2, "parse_mode": "HTML"})

msg_t = format_whatsapp_report(output_data ,'Treading Stocks')
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
             params={"chat_id": CHAT_ID, "text": msg_t, "parse_mode": "HTML"})


msg_p0 = format_whatsapp_report(priority0_data,'Least Priority Stocks')
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
            params={"chat_id": CHAT_ID, "text": msg_p0, "parse_mode": "HTML"})



<Response [200]>

In [97]:
error_msg = format_whatsapp_error(error_data,'Error Stocks')
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
            params={"chat_id": TEST_ID, "text": error_msg, "parse_mode": "HTML"})

msg_40 = format_whatsapp_report(rsi_40_cross_data,'RSI 40 Crossover Stocks')
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
                        params={"chat_id": TEST_ID, "text": msg_40, "parse_mode": "HTML"})

<Response [200]>

In [98]:
print(error_data)

[{'Name': 'ISEC', 'Token': 2489, 'Setup_RSI': 69, 'reasone': 'Date from API 2025-03-21, processing date 2025-11-06'}]


In [99]:
# # Create Files For the output

# todays_date = datetime.today().strftime("%Y-%m-%d")
# output_df = pd.DataFrame(output_data)
# output_df.to_csv(f'Daily_Report/Trending/RSI-Setup-treading-{todays_date}.csv', index=False)
# error_df = pd.DataFrame(error_data)
# error_df.to_csv(f'Daily_Report/Error/RSI-Setup-error-{todays_date}.csv', index=False)
# priority_df = pd.DataFrame(priority_data)
# priority_df.to_csv(f'Daily_Report/Priority/RSI-Setup-Priority-{todays_date}.csv', index=False)